# 05 — Multilingual Intelligence

**NewsBot Intelligence System 2.0** | ITAI 2373 | Trilok Kalani (SOLO)

Detect language, translate to English, and run the standard pipeline on non-English text.

In [1]:
# --- Setup: works in Colab and locally ---
import os, sys, subprocess

def find_repo_root(start="."):
    p = os.path.abspath(start)
    for _ in range(6):
        if os.path.isdir(os.path.join(p, "src")) and os.path.exists(os.path.join(p, "src", "newsbot.py")):
            return p
        p = os.path.dirname(p)
    return None

ROOT = find_repo_root()
if ROOT is None:
    # Running on a fresh Colab: clone the repo
    if not os.path.isdir("ITAI2373-Portfolio"):
        subprocess.run(["git","clone","--depth","1",
                        "https://github.com/Tikskalani/ITAI2373-Portfolio.git"], check=False)
    ROOT = find_repo_root("ITAI2373-Portfolio/ITAI2373-NewsBot-Final") or \
           find_repo_root("ITAI2373-Portfolio")
sys.path.insert(0, ROOT)
print("Repo root:", ROOT)

# spaCy model (quiet no-op if already present)
try:
    import spacy; spacy.load("en_core_web_sm")
except Exception:
    subprocess.run([sys.executable,"-m","spacy","download","en_core_web_sm"], check=False)

import pandas as pd
DATA = os.path.join(ROOT, "data", "raw", "newsbot_bbc.csv")
df = pd.read_csv(DATA)
print("Loaded", len(df), "articles;", df["category"].nunique(), "categories")
df.head(2)

Repo root: /content/ITAI2373-NewsBot-Final


Loaded 2225 articles; 5 categories


,article_id,category,text
0,business_001,business,Ad sales boost Time Warner profit Quarterly pr...
1,business_002,business,Dollar gains on Greenspan speech The dollar ha...


### Language detection

In [2]:
from src.multilingual.language_detector import LanguageDetector
ld = LanguageDetector()
for t in ["El equipo ganó la final de la copa.",
          "Le gouvernement a annoncé de nouvelles mesures.",
          "The company reported record profits."]:
    print(ld.detect(t), "-", t)

es - El equipo ganó la final de la copa.
fr - Le gouvernement a annoncé de nouvelles mesures.
en - The company reported record profits.


### Translate and analyze in English
Translation uses a free online backend, so this cell needs a network connection. It is wrapped so the notebook still runs offline.

In [3]:
from src.multilingual.cross_lingual_analyzer import CrossLingualAnalyzer
from src.newsbot import NewsBot
bot = NewsBot().train(df["text"], df["category"])
cla = CrossLingualAnalyzer()
try:
    res = cla.to_english("El banco central subió las tasas de interés para frenar la inflación.")
    print("Detected:", res["detected_language"])
    print("English :", res["translated_text"])
    print("Analysis:", bot.analyze(res["translated_text"])["classification"])
except Exception as e:
    print("Translation needs a network connection; skipped:", type(e).__name__)

Translation needs a network connection; skipped: KeyError


**Takeaway.** Non-English coverage is detected, translated to English, and then handled by the same trusted pipeline, rather than assuming English-trained models read every language.